In [2]:
from pyspark.sql import SparkSession
import pandas as pd

In [3]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

## Step 2: Load and Explore the Dataset

In [4]:
spark = SparkSession.builder.appName("JobPostingsAnalysis").getOrCreate()

df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("data/lightcast_job_postings.csv")

df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 18:15:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATION_LEVELS_NAME: string (nullable = true)
 |-- MIN_EDULEVELS: integer (

In [7]:
pdf = pd.read_csv("data/lightcast_job_postings.csv", nrows=10000)
pdf.info()

/tmp/ipykernel_3470/1066398164.py:1: DtypeWarning: Columns (0: COMPANY_IS_STAFFING, 1: IS_INTERNSHIP) have mixed types. Specify dtype option on import or set low_memory=False.
  pdf = pd.read_csv("data/lightcast_job_postings.csv", nrows=10000)


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 131 entries, ID to NAICS_2022_6_NAME
dtypes: float64(38), object(2), str(91)
memory usage: 98.9+ MB


In [8]:
pdf.head()

,ID,LAST_UPDATED_DATE,LAST_UPDATED_TIMESTAMP,DUPLICATES,POSTED,EXPIRED,DURATION,SOURCE_TYPES,SOURCES,URL,...,NAICS_2022_2,NAICS_2022_2_NAME,NAICS_2022_3,NAICS_2022_3_NAME,NAICS_2022_4,NAICS_2022_4_NAME,NAICS_2022_5,NAICS_2022_5_NAME,NAICS_2022_6,NAICS_2022_6_NAME
0,1f57d95acf4dc67ed2819eb12f049f6a5c11782c,9/6/2024,2024-09-06 20:32:57.352 Z,0.0,6/2/2024,6/8/2024,6.0,"[\n ""Company""\n]","[\n ""brassring.com""\n]","[\n ""https://sjobs.brassring.com/TGnewUI/Sear...",...,44.0,Retail Trade,441.0,Motor Vehicle and Parts Dealers,4413.0,"Automotive Parts, Accessories, and Tire Retailers",44133.0,Automotive Parts and Accessories Retailers,441330.0,Automotive Parts and Accessories Retailers
1,0cb072af26757b6c4ea9464472a50a443af681ac,8/2/2024,2024-08-02 17:08:58.838 Z,0.0,6/2/2024,8/1/2024,NaN,"[\n ""Job Board""\n]","[\n ""maine.gov""\n]","[\n ""https://joblink.maine.gov/jobs/1085740""\n]",...,56.0,Administrative and Support and Waste Managemen...,561.0,Administrative and Support Services,5613.0,Employment Services,56132.0,Temporary Help Services,561320.0,Temporary Help Services
2,85318b12b3331fa490d32ad014379df01855c557,9/6/2024,2024-09-06 20:32:57.352 Z,1.0,6/2/2024,7/7/2024,35.0,"[\n ""Job Board""\n]","[\n ""dejobs.org""\n]","[\n ""https://dejobs.org/dallas-tx/data-analys...",...,52.0,Finance and Insurance,524.0,Insurance Carriers and Related Activities,5242.0,"Agencies, Brokerages, and Other Insurance Rela...",52429.0,Other Insurance Related Activities,524291.0,Claims Adjusting
3,1b5c3941e54a1889ef4f8ae55b401a550708a310,9/6/2024,2024-09-06 20:32:57.352 Z,1.0,6/2/2024,7/20/2024,48.0,"[\n ""Job Board""\n]","[\n ""disabledperson.com"",\n ""dejobs.org""\n]","[\n ""https://www.disabledperson.com/jobs/5948...",...,52.0,Finance and Insurance,522.0,Credit Intermediation and Related Activities,5221.0,Depository Credit Intermediation,52211.0,Commercial Banking,522110.0,Commercial Banking
4,cb5ca25f02bdf25c13edfede7931508bfd9e858f,6/19/2024,2024-06-19 07:00:00.000 Z,0.0,6/2/2024,6/17/2024,15.0,"[\n ""FreeJobBoard""\n]","[\n ""craigslist.org""\n]","[\n ""https://modesto.craigslist.org/sls/77475...",...,99.0,Unclassified Industry,999.0,Unclassified Industry,9999.0,Unclassified Industry,99999.0,Unclassified Industry,999999.0,Unclassified Industry


## Step 3: Data Cleaning

In [9]:
pdf = pdf.drop_duplicates()

median_salary_from = pdf['SALARY_FROM'].median()
median_salary_to = pdf['SALARY_TO'].median()
pdf['SALARY_FROM'] = pdf['SALARY_FROM'].fillna(median_salary_from)
pdf['SALARY_TO'] = pdf['SALARY_TO'].fillna(median_salary_to)

pdf['POSTED'] = pd.to_datetime(pdf['POSTED'], errors='coerce')
pdf['EXPIRED'] = pd.to_datetime(pdf['EXPIRED'], errors='coerce')

print("Cleaning complete.")
print(pdf.shape)

Cleaning complete.
(9995, 131)


## Step 4: Exploratory Data Analysis

### Total Job Postings

In [10]:
print(f"Total job postings: {len(pdf)}")

Total job postings: 9995


### Top 5 Job Titles

In [13]:
import plotly.express as px

top5 = pdf['TITLE_CLEAN'].value_counts().head(5).reset_index()
top5.columns = ['Job Title', 'Count']

fig = px.bar(top5.sort_values('Count'), x='Count', y='Job Title',
             orientation='h', title='Top 5 Most Common Job Titles',
             labels={'Count': 'Number of Postings', 'Job Title': 'Job Title'})
fig.update_traces(marker_color='steelblue')
fig.show()

### Average Salary by Employment Type

In [17]:
avg_salary = pdf.groupby('EMPLOYMENT_TYPE_NAME')[['SALARY_FROM','SALARY_TO']].mean().round(2).reset_index()

fig2 = px.bar(avg_salary, x='EMPLOYMENT_TYPE_NAME', y='SALARY_FROM',
              title='Average Salary by Employment Type',
              labels={'EMPLOYMENT_TYPE_NAME': 'Employment Type', 'SALARY_FROM': 'Avg Salary ($)'})
fig2.update_traces(marker_color='steelblue')
fig2.show()

### State with Most Job Postings

In [20]:
state_counts = pdf['STATE_NAME'].value_counts().reset_index()
state_counts.columns = ['State', 'Count']

top_state = state_counts.iloc[0]
print(f"State with most job postings: {top_state['State']} ({top_state['Count']:,} postings)")

fig3 = px.bar(state_counts.head(10), x='State', y='Count',
              title='Top 10 States by Job Postings',
              labels={'State': 'State', 'Count': 'Number of Postings'})
fig3.update_traces(marker_color='steelblue')
fig3.show()

State with most job postings: Texas (1,161 postings)
